# Challenge 3: Human-in-the-Loop & Resilience

## Why HITL Matters

Your workflow routes correctly — but would you trust it to `restart_pod` in production
at 3 AM without asking anyone?

MAF provides:
- `ctx.request_info()` — pauses the workflow and asks the caller for input
- `@tool(approval_mode="always_require")` — tool calls trigger approval events
- `workflow.run(responses={...})` — resumes a paused workflow with answers
- `@response_handler` — processes the human's response and continues

## What You'll Build

| Feature | MAF API | Purpose |
|---------|---------|--------|
| Explicit HITL gate | `ctx.request_info(data, type)` | Pause workflow, show data to human |
| Response handler | `@response_handler` | Process the human's answer |
| Resume | `workflow.run(responses={request_id: answer})` | Continue after human decision |
| Tool approval | `@tool(approval_mode="always_require")` | Per-tool-call approval |
| Functional HITL | `@workflow` + `ctx.request_info()` | Simpler pattern for linear flows |

## Setup

Load all MAF imports (`Executor`, `WorkflowBuilder`, `WorkflowContext`, etc.), the `FoundryChatClient` for connecting to Azure AI Foundry, and the mock infrastructure tools (`restart_pod`, `scale_service`, etc.) that simulate real ops actions.

In [1]:
import os
import sys
import json
from typing import Any

sys.path.insert(0, "..")
from dotenv import load_dotenv

from agent_framework import (
    Agent, AgentExecutor, AgentExecutorRequest, AgentExecutorResponse,
    Executor, Message, WorkflowBuilder, WorkflowContext, WorkflowRunState,
    executor, handler, response_handler, tool,
)
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatOptions
from azure.identity import AzureCliCredential

from tools.mock_infra import restart_pod, scale_service, flush_cache, toggle_feature_flag

load_dotenv("../.env")
print("\u2705 Imports ready")

c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Github Repo\maf-lab\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


✅ Imports ready


---
## Step 1: The HITL Pattern

1. Executor calls `await ctx.request_info(data, type)` → workflow **PAUSES**
2. Caller checks `result.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS`
3. Caller gets `result.get_request_info_events()` to see what was asked
4. Caller resumes with `workflow.run(responses={request_id: answer})`
5. `@response_handler` method receives the answer

**Critical:** `@response_handler` signature MUST be `(self, original_request, response, ctx)` — 4 params

---
## Step 2: Approval Gate Executor (REFERENCE)

Below we define two executors that work together:

- **`ApprovalGate`** — When it receives a message (e.g. "restart pod-3"), it saves the action in workflow state, then calls `ctx.request_info()` which **pauses the entire workflow** and surfaces a prompt to the caller. The `@response_handler` method fires when the human responds, forwarding their answer to the next executor.

- **`ExecuteAction`** — Receives the human's response. If they said "yes", it executes the action; otherwise it aborts. This is the terminal node that produces the final output via `ctx.yield_output()`.

In [2]:
class ApprovalGate(Executor):
    """Pauses the workflow to ask a human for approval."""
    def __init__(self):
        super().__init__(id="approval-gate")
    
    @handler
    async def handle(self, message: str, ctx: WorkflowContext) -> None:
        ctx.set_state("pending_action", message)
        # PAUSES the workflow here:
        approval = await ctx.request_info(
            f"\U0001f6a8 APPROVAL REQUIRED:\n{message}\n\nApprove? (yes/no)",
            str,
        )
        await ctx.send_message(approval)
    
    @response_handler
    async def on_response(self, original_request: str, response: str, ctx: WorkflowContext) -> None:
        """4 params required: self, original_request, response, ctx"""
        await ctx.send_message(response)

print("\u2705 ApprovalGate defined")

✅ ApprovalGate defined


In [3]:
class ExecuteAction(Executor):
    """Executes or aborts based on approval."""
    def __init__(self):
        super().__init__(id="execute-action")
    
    @handler
    async def handle(self, message: str, ctx: WorkflowContext) -> None:
        action = ctx.get_state("pending_action")
        if "yes" in message.lower():
            await ctx.yield_output(f"\u2705 APPROVED — Executing: {action}")
        else:
            await ctx.yield_output(f"\u274c REJECTED — Aborted: {action}")

print("\u2705 ExecuteAction defined")

✅ ExecuteAction defined


---
## Step 3: Wire and Run the HITL Workflow

Now we connect the two executors into a workflow graph (`ApprovalGate → ExecuteAction`) and run it. Watch what happens:

1. **First `workflow.run()`** — The workflow starts, hits `ctx.request_info()` inside `ApprovalGate`, and **pauses** with state `IDLE_WITH_PENDING_REQUESTS`
2. **Read the pending request** — We inspect `get_request_info_events()` to see what the workflow is asking the human
3. **Second `workflow.run(responses=...)`** — We resume the workflow by providing the human's answer ("yes"), which triggers `@response_handler` → `ExecuteAction` → final output

In [4]:
gate = ApprovalGate()
execute = ExecuteAction()

# Build the HITL workflow: ApprovalGate → ExecuteAction
workflow = (WorkflowBuilder(start_executor=gate)
    .add_edge(gate, execute)
    .build())

# Run — this will PAUSE at ctx.request_info()
result = await workflow.run("restart payment-api pod-3")
print(f"State: {result.get_final_state()}")

# Get the pending request
events = result.get_request_info_events()
req_id = events[0].request_id
print(f"Human sees: {events[0].data}")

# Resume with approval
result2 = await workflow.run(responses={req_id: "yes"})
print(f"Output: {result2.get_outputs()[0]}")

C:\Users\kiranpanchal\AppData\Local\Temp\ipykernel_29132\3016686008.py:7: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build())
Executor 'approval-gate' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowContext[T] generics in handlers for better validation.


State: WorkflowRunState.IDLE_WITH_PENDING_REQUESTS
Human sees: 🚨 APPROVAL REQUIRED:
restart payment-api pod-3

Approve? (yes/no)
Output: ✅ APPROVED — Executing: restart payment-api pod-3


**Validation** — Confirms the workflow paused correctly (`IDLE_WITH_PENDING_REQUESTS`), resumed after approval (`IDLE`), and produced the expected "APPROVED" output.

In [5]:
# Validate
assert result.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS
assert result2.get_final_state() == WorkflowRunState.IDLE
assert "APPROVED" in result2.get_outputs()[0]
print("\u2705 HITL workflow works!")
print(f"   Output: {result2.get_outputs()[0]}")

✅ HITL workflow works!
   Output: ✅ APPROVED — Executing: restart payment-api pod-3


---
## Step 4: Tool Approval with Agent

A more realistic pattern: instead of manually pausing, we let an **agent call tools that require approval**. The tools (`restart_pod`, `scale_service`, etc.) are decorated with `approval_mode="always_require"`, so MAF automatically pauses the workflow whenever the agent tries to call one.

The **approval loop** below handles this — it keeps checking if the workflow is paused for approval, auto-approves each request, and resumes until the workflow completes.

```python
while events.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS:
    for evt in events.get_request_info_events():
        responses[evt.request_id] = evt.data.to_function_approval_response(approved=True)
    events = await workflow.run(responses=responses)
```

The workflow graph is: `prepare → AgentExecutor(remediation_agent) → done`

In [6]:
client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

remediation_agent = Agent(
    client,
    id="remediation-executor",
    name="RemediationExecutor",
    instructions="Execute remediation. Call restart_pod, scale_service, flush_cache, or toggle_feature_flag as needed.",
    tools=[restart_pod, scale_service, flush_cache, toggle_feature_flag],
)

# Build workflow with AgentExecutor + approval loop
@executor(id="prepare")
async def prepare(plan: str, ctx: WorkflowContext) -> None:
    msg = Message("user", contents=[plan])
    await ctx.send_message(AgentExecutorRequest(messages=[msg], should_respond=True))

@executor(id="done")
async def done(response: AgentExecutorResponse, ctx: WorkflowContext) -> None:
    await ctx.yield_output(f"Done: {response.agent_response.text[:200]}")

agent_exec = AgentExecutor(remediation_agent)
tool_workflow = (WorkflowBuilder(start_executor=prepare)
    .add_edge(prepare, agent_exec)
    .add_edge(agent_exec, done)
    .build())

# Run with approval loop
events = await tool_workflow.run("restart_pod for payment-api pod-3, reason: OOM")
while events.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS:
    responses = {}
    for evt in events.get_request_info_events():
        responses[evt.request_id] = evt.data.to_function_approval_response(approved=True)
    events = await tool_workflow.run(responses=responses)
print(events.get_outputs())

C:\Users\kiranpanchal\AppData\Local\Temp\ipykernel_29132\584284726.py:29: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build())
Executor 'prepare' has no output type annotations. Type compatibility validation will be skipped for edges from this executor. Consider adding WorkflowContext[T] generics in handlers for better validation.
Response resp_0a913179f035c58c006a46d4c500d881979c2ed7ba92b6db25 contains 1 user input requests but total message contents are 2. This indicates the response contains both user input requests and message contents. Double check if this is the intended behavior, as non user input request contents in this response will not be emitted.


[<agent_framework._types.AgentResponse object at 0x0000022B7B8F4B00>, 'Done: The pod pod-3 of the payment-api service has been restarted due to an Out Of Memory (OOM) issue. The restart was successful and the pod is now healthy, with a brief downtime of 8 seconds.\n\nIf you need']


---
## Step 5: Functional Workflow with `@workflow`

MAF also offers a **simpler pattern** for linear flows. Instead of defining `Executor` classes and wiring a graph, you write a plain `async` function decorated with `@workflow` and call `ctx.request_info()` directly inside it.

Below, the workflow pauses to ask for approval with a custom `request_id` ("plan_approval"). The caller resumes by passing `responses={"plan_approval": "yes"}`. This is ideal when your flow is sequential and doesn't need complex graph routing.

In [7]:
from agent_framework import RunContext, workflow, step

@workflow
async def remediate_with_approval(plan: str, ctx: RunContext) -> str:
    approval = await ctx.request_info(
        f"Plan: {plan}\nApprove?",
        response_type=str,
        request_id="plan_approval",
    )
    if "yes" in approval.lower():
        return f"Executing: {plan}"
    return f"Aborted: {plan}"

r1 = await remediate_with_approval.run("restart pod-3")
assert r1.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS
r2 = await remediate_with_approval.run(responses={"plan_approval": "yes"})
print(r2.get_outputs())

['Executing: restart pod-3']


C:\Users\kiranpanchal\AppData\Local\Temp\ipykernel_29132\473887203.py:3: ExperimentalWarning: [FUNCTIONAL_WORKFLOWS] workflow is experimental and may change or be removed in future versions without notice.


---
## Summary

| Pattern | API | When to Use |
|---------|-----|-------------|
| Explicit pause | `ctx.request_info()` + `@response_handler` | Business decisions, plan review |
| Tool-level approval | `@tool(approval_mode="always_require")` | Dangerous tool calls |
| Workflow resume | `workflow.run(responses={...})` | All HITL patterns |
| Functional HITL | `@workflow` + `ctx.request_info()` | Simple linear flows |

---
## ➡️ Bonus: Challenge 4

[Open Challenge 4 →](../challenge-4/README.md)